# Week 2 - Day 1: Chaining and Routing

> **How this notebook works.** Each **TODO** cell is real code with a few blanks: `___`. Replace every `___` (the comment on that line tells you what goes there), run the cell, then run the **check** below it and compare what it prints with the `# expected:` comments. After each part there is **Classwork**: a new problem that you build **yourself**, from an empty cell. Use the code above as your guide, but type every line yourself; open the hint only if you are stuck. Finished early? Try the *If you finish early* idea under it.

In Week 1 you built an **agent**: the model chose its own tools and steps. This week you build **workflows**: **you** draw the steps, and the model does one job in each step. The code follows the official LangGraph guide **[Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)**.

Today:
1. **The augmented LLM**: a model that answers in a fixed format, and can ask for a tool.
2. **Prompt chaining**: steps in a fixed order, with a check (a "gate") between them.
3. **Routing**: one step reads the question and chooses the path.

## Setup
You need **two** free keys, the same as Week 1:
- **Groq** (console.groq.com -> API Keys), saved as `GROQ_API_KEY`
- **Tavily** (tavily.com), saved as `TAVILY_API_KEY`

**On Google Colab:** click the key icon (Secrets) on the left, add both keys, and switch **Notebook access** on. **On your computer:** put them in a `.env` file.

> **Tips.** The free Groq plan allows about 8,000 tokens a minute. If a cell pauses, it is waiting for the limit. If a cell ends with `RateLimitError`, wait a minute and run it again. If a cell ends with `NameError: name '___' is not defined`, a blank is still empty. If a check ends with another `NameError`, run the TODO cell above it first.

In [ ]:
# What: installs the libraries for this notebook (about 1 minute, once per Colab session).
# Why:  Colab starts without LangChain, LangGraph or the Groq and Tavily connectors.
# How:  %pip is the notebook command for pip: -q keeps it quiet, -U gets the newest versions.
%pip install -qU langchain langgraph langchain-groq langchain-tavily python-dotenv

In [ ]:
# What: puts your API keys into the environment, where LangChain looks for them.
# Why:  keys are secrets: they never go into the notebook text, so the notebook can be shared safely.
# How:  on Colab, userdata reads the Secrets panel; on your computer, load_dotenv reads a .env file.
import os
from dotenv import load_dotenv

load_dotenv()                                                   # on your computer: reads the keys from a .env file
try:
    from google.colab import userdata                           # on Google Colab: reads the keys from Secrets
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except ImportError:
    pass

print("Groq key found:", "GROQ_API_KEY" in os.environ)
print("Tavily key found:", "TAVILY_API_KEY" in os.environ)

In [ ]:
# What: the chat model for the whole week: gpt-oss-120b, hosted by Groq.
# Why:  one model object, created once, is used by every step below.
# How:  init_chat_model("provider:model") picks the right connector. temperature=0 gives the most repeatable answers.
#       max_retries=10: when the free plan's speed limit is reached, it waits and tries again instead of failing.
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0, max_retries=10)

response = llm.invoke("In one sentence, what is a design pattern in software?")     # send one message, get one reply
print(response.content)                 # the text of the reply
print(response.usage_metadata)          # the tokens this call used

---
# Part 1 - The augmented LLM

A workflow is built from a model with two extras:
- **Structured output**: the model must answer by filling a small class. Your code can then read the answer, for example `answer.step`.
- **Tools**: the model can ask for a tool, such as a web search.

Read the class below like a form with one field:
- `step` is the field.
- `Literal["search", "direct"]` means: only these two words are allowed.
- `description` tells the model when to pick each word. **The model reads it**, so write it clearly.

In [ ]:
# TODO 1: fill the 2 blanks.
# What: a "form" (class) with one field, step, and a router model that must fill it in.
# Why:  the graph needs a fixed word ("search" or "direct") that an if can check. Free text could say anything.
# How:  Literal lists the only allowed words. The description tells the model when to use each word.
#       with_structured_output makes the model answer by filling the class (strict=True: exactly this format).
from typing import Literal                          # Literal: only the listed words are allowed
from pydantic import BaseModel, Field               # BaseModel + Field: the classes for structured output

class Route(BaseModel):
    step: Literal["search", ___] = Field(                      # the second allowed word: "direct"
        description="search: the question needs facts from the web: news, prices, products to buy, weather, anything 'right now', or advice that depends on this year, such as 'Is it worth learning Rust in 2026?'. direct: common knowledge that does not change, or a greeting."
    )

router = llm.with_structured_output(___, method="json_schema", strict=True)     # the class the model must fill

In [ ]:
# CHECK 1: compare with the expected words.
# How it works: router.invoke(question) returns a filled-in Route; .step reads its field.
print(router.invoke("What is the weather in Pokhara today?").step)     # expected: search
print(router.invoke("Is it worth learning Rust in 2026?").step)        # expected: search
print(router.invoke("What is a database?").step)                       # expected: direct
print(router.invoke("Hi, how are you?").step)                          # expected: direct

If a word is wrong, make the `description` in `Route` clearer and run both cells again.

### A model with a tool
`TavilySearch` is a ready-made web search tool. `bind_tools` tells the model the tool exists. The model does **not** run the tool: it only **asks** for it, with the search words it wants. Run the cell and look at `tool_calls`.

In [ ]:
# What: a web search tool, and a model that knows the tool exists.
# Why:  a model alone only knows its training data; a tool lets it ask for fresh facts.
# How:  bind_tools sends the tool's name and description with every call. The reply contains a tool_call
#       (the tool name + search words). The model does NOT run the search: your code or an agent does.
from langchain_tavily import TavilySearch           # the ready-made web search tool

search = TavilySearch(max_results=2)                      # a ready-made web search tool
llm_with_tools = llm.bind_tools([search])

msg = llm_with_tools.invoke("What is the latest news about electric cars in Nepal?")
print(msg.tool_calls)                                     # expected: a request for tavily_search, with search words

## Classwork 1 · A review sorter (about 7 minutes)
**Build it yourself** in the empty cell below. Use TODO 1 as your guide, but type every line yourself.

A shop gets hundreds of reviews a day. Build a model that sorts each review into `positive`, `negative` or `neutral`.

**Steps**
1. Write a class `Review` with one field, `mood`, that allows only `"positive"`, `"negative"` or `"neutral"`. Give the field a description that tells the model when to pick each word.
2. Make `sorter`: `llm` with structured output, using your class.
3. Run the check below.

<details><summary><b>Stuck? Open a hint</b></summary>

It is the same shape as `Route` in TODO 1: `class Review(BaseModel):`, then one line `mood: Literal[...] = Field(description="...")`, then `sorter = llm.with_structured_output(Review, method="json_schema", strict=True)`.
</details>

In [ ]:
# CLASSWORK 1: the class Review and the model sorter (the check below uses these names).

# Write your code here.

In [ ]:
# CHECK: your sorter on three reviews.
for text in ["The delivery was fast and the shoes fit perfectly!",
             "The phone stopped working after two days. I want my money back.",
             "Do you also sell this jacket in blue?"]:
    print(sorter.invoke(text).mood, "<-", text)
# expected: positive, negative, neutral

**If you finish early:** add a fourth word, `mixed`, for reviews that are happy and unhappy at once, and test it on *"Great camera, but the battery is terrible."*

---
# Part 2 - Prompt chaining

A **chain** is a list of steps: the output of one step is the input of the next. Our chain writes an answer. A gate counts its words. If the answer is too long, one more step makes it shorter.

```
question --> write the answer --> more than 80 words? --no--> answer
                                          |
                                         yes
                                          v
                                  shorten the answer --> answer
```

You build **the same chain twice**:
- **A) with LangChain**: the pipe `|`, the shortest way to connect steps in a line,
- **B) with LangGraph**: the same steps as a graph, where every step and the gate are visible.

## Part 2A - The chain with LangChain: the pipe `|`
Three LangChain pieces:
- `ChatPromptTemplate.from_template("... {question}")`: a prompt with a **placeholder** in curly brackets. `invoke({"question": "..."})` fills it in.
- `llm`: the model, as before.
- `StrOutputParser()`: the model returns a message object; the parser keeps **only its text**.

`a | b` means: **the output of `a` goes into `b`**. So `prompt | llm | StrOutputParser()` is one small chain: fill the prompt, ask the model, keep the text. A plain Python function can be a step too: here, the gate.

In [ ]:
# TODO 2: the chain with LangChain. Fill the 3 blanks.
# What: write an answer, check its length, and shorten it if it has more than 80 words.
# Why:  for steps in a straight line, LangChain's pipe is the shortest way to connect them.
# How:  each | passes the output of the left step into the right step. The gate is a plain Python function:
#       inside a pipe, LangChain turns it into a step automatically.
from langchain_core.prompts import ChatPromptTemplate       # a prompt with {placeholders}
from langchain_core.output_parsers import StrOutputParser   # turns the model's reply into plain text

write_prompt = ChatPromptTemplate.from_template("Answer this question: {question}")
shorten_prompt = ChatPromptTemplate.from_template("Shorten this to at most 3 sentences: {answer}")

write_step = write_prompt | llm | ___          # prompt -> model -> text (the parser, with brackets)
shorten_step = shorten_prompt | ___ | StrOutputParser()      # the same three parts: the model goes in the middle

def check_and_shorten(answer):                       # the gate: plain Python, no model call
    if len(answer.split()) > 80:                     # more than 80 words?
        return shorten_step.invoke({___: answer})   # fill the placeholder of shorten_prompt: "answer"
    return answer                                    # short enough: keep it as it is

lc_chain = write_step | check_and_shorten            # write, then the gate

In [ ]:
# CHECK 2: run the LangChain chain.
# How it works: invoke fills {question}, runs every step of the pipe, and returns only the LAST output (a text).
answer = lc_chain.invoke({"question": "Explain how the internet works."})
print(len(answer.split()), "words:", answer)        # expected: a short answer of about 3 sentences (the first one was much longer)

Notice what you **cannot** see: how long was the first answer? The pipe only gives back the final text; the first answer is gone. Part 2B keeps every step in a **state**.

## Part 2B - The same chain with LangGraph
LangGraph needs four words:

| Word | Meaning |
|---|---|
| **state** | a dictionary that every step can read |
| **node** | a function: it reads the state and returns **only the keys it changes** |
| **edge** | which node runs next |
| **gate** | a small function that chooses the next node (a "conditional edge") |

The same chain as a graph:

```
START --> write_answer --> check_length --"Pass"--> END
                                |
                              "Fail"
                                v
                             shorten --> END
```

In [ ]:
# TODO 3: the state and the three functions. Fill the 2 blanks.
# What: the state (the shared dictionary) and the three functions of the chain.
# Why:  every node reads the state and returns only the keys it changes; LangGraph merges them in.
# How:  TypedDict lists the keys. write_answer and shorten call the model. check_length is plain Python:
#       it returns a word ("Pass" or "Fail") that the graph uses to choose the next node.
from typing_extensions import TypedDict             # TypedDict: describes the keys of a state

class ChainState(TypedDict):
    question: str
    answer: str
    short_answer: str

def write_answer(state: ChainState):
    msg = llm.invoke(f"Answer this question: {state['question']}")
    return {"answer": msg.content}                 # a node returns only the key it changes

def check_length(state: ChainState):               # the gate: plain Python, no model call
    if len(state["answer"].split()) > ___:         # the word limit: 80
        return "Fail"
    return "Pass"

def shorten(state: ChainState):
    msg = llm.invoke(f"Shorten this to at most 3 sentences: {state['answer']}")
    return {___: msg.content}          # the key that shorten fills: "short_answer"

In [ ]:
# TODO 4: build the graph. Fill the 2 blanks.
# What: the graph: which node runs first, and what comes after each node.
# Why:  the functions alone do nothing; the graph says in which order they run.
# How:  add_node gives each function a name. add_edge = always go there next.
#       add_conditional_edges = run the gate function, and use its word to pick the next node.
#       compile() checks the graph and turns it into something you can invoke.
from langgraph.graph import StateGraph, START, END  # the graph builder, and its start and end points

builder = StateGraph(ChainState)
builder.add_node("write_answer", write_answer)
builder.add_node("shorten", shorten)
builder.add_edge(START, ___)                                                 # the first node
builder.add_conditional_edges("write_answer", check_length, {"Fail": ___, "Pass": END})   # where a long answer goes
builder.add_edge("shorten", END)
chain = builder.compile()

In [ ]:
# Draw the graph: it should look like the picture above.
# How: get_graph() reads the compiled graph; draw_mermaid_png() turns it into a picture (it needs internet).
from IPython.display import Image, display          # shows the graph picture in the notebook
display(Image(chain.get_graph().draw_mermaid_png()))

In [ ]:
# CHECK 3: a question with a long answer.
# How it works: invoke runs the whole graph from a starting state and returns the final state (a dictionary).
state = chain.invoke({"question": "Explain how the internet works."})
print("Words in the first answer:", len(state["answer"].split()))     # expected: more than 80
print("Short answer:", state["short_answer"])                         # expected: at most 3 sentences

A `KeyError: 'short_answer'` means `shorten` did not run: check the gate and the conditional edge.

## Classwork 2 · A tweet chain with the pipe (about 10 minutes)
**Build it yourself** in the empty cell below, with the LangChain pipe. Use TODO 2 as your guide, but type every line yourself.

Write a tweet about a topic. If the tweet has more than 200 characters, a second step shortens it. (`len(text)` counts characters.)

**Steps**
1. Two prompts with `ChatPromptTemplate.from_template`: one that writes a tweet about `{topic}` with 3 hashtags, and one that rewrites `{tweet}` in at most 150 characters.
2. Two small chains, each `prompt | llm | StrOutputParser()`.
3. A gate function: if the tweet has more than 200 characters, run the shorten chain; otherwise return the tweet.
4. Join them: `tweet_chain = your_write_chain | your_gate`.

<details><summary><b>Stuck? Open a hint</b></summary>

Copy the shape of TODO 2 and change three things: the two prompt texts, the placeholder names (`{topic}` and `{tweet}`), and the gate: `len(answer.split()) > 80` becomes `len(tweet) > 200`. The key in `invoke({...})` must be the same as the placeholder.
</details>

In [ ]:
# CLASSWORK 2: the chain tweet_chain (input: {"topic": ...}, output: the final tweet as text).

# Write your code here.

In [ ]:
# CHECK: run your chain.
tweet = tweet_chain.invoke({"topic": "why students should learn Python"})
print(len(tweet), "characters:", tweet)       # expected: about 200 characters or fewer, with hashtags

**If you finish early:** build the same tweet chain as a LangGraph graph (like TODO 3 and TODO 4), and print the first tweet **and** the short one. The pipe could not show you both.

---
# Part 3 - Routing

A **router** reads the question and chooses the path. Your `router` from Part 1 does exactly that. Now it becomes a graph:

```
                       +--"search"--> answer_search  (an agent with web search) --> END
START --> route_question
                       +--"direct"--> answer_direct  (one model call)            --> END
```

The **search** path is an **agent**, like Week 1: `create_agent` with the search tool. The **direct** path is one cheap model call.

In [ ]:
# TODO 5: the agent, the state and the three nodes. Fill the 2 blanks.
# What: a search agent, the router state, and the three nodes of the router graph.
# Why:  each path is one node: route_question only decides, the other two answer.
# How:  create_agent (Week 1) builds an agent that can call the search tool.
#       route_question stores the router's word in the state. answer_search gives the question to the agent
#       and keeps its last message. answer_direct is one plain model call.
from langchain.agents import create_agent           # builds an agent, as in Week 1

search_agent = create_agent(llm, tools=[___], system_prompt="Call the tavily_search tool once (it is your only tool), then answer in 3 sentences and give the link.")   # the tool from Part 1

class RouterState(TypedDict):
    question: str
    decision: str
    answer: str

def route_question(state: RouterState):
    route = router.invoke(state["question"])
    return {"decision": route.___}                              # the field of Route

def answer_search(state: RouterState):
    result = search_agent.invoke({"messages": state["question"]})
    return {"answer": result["messages"][-1].content}          # the last message is the agent's answer

def answer_direct(state: RouterState):
    msg = llm.invoke(state["question"])
    return {"answer": msg.content}

In [ ]:
# TODO 6: the gate after route_question. It returns the NAME of the next node. Fill the 2 blanks.
# What: the gate after route_question.
# Why:  the router says a word ("search"), but the graph needs a node name ("answer_search").
# How:  read the word from the state with state["decision"], and return the matching node name.
def route_decision(state: RouterState):
    if state["decision"] == ___:                # the router's word for the web
        return ___                       # the node for that word
    return "answer_direct"

In [ ]:
# TODO 7: build the router graph. Fill the 2 blanks.
# What: the router graph: START -> route_question -> answer_search or answer_direct -> END.
# Why:  this is where the router's choice becomes a path.
# How:  the dictionary in add_conditional_edges maps each word the gate returns to a node.
builder = StateGraph(RouterState)
builder.add_node("route_question", route_question)
builder.add_node("answer_search", answer_search)
builder.add_node("answer_direct", answer_direct)
builder.add_edge(START, ___)                                   # the first node
builder.add_conditional_edges("route_question", route_decision, {"answer_search": "answer_search", "answer_direct": "answer_direct"})
builder.add_edge("answer_search", END)
builder.add_edge("answer_direct", ___)                                      # where the direct path ends
router_workflow = builder.compile()

In [ ]:
# Draw the router graph: one node in, two paths out.
display(Image(router_workflow.get_graph().draw_mermaid_png()))

In [ ]:
# CHECK 4: one question for each path. get_usage_metadata_callback counts the tokens of every call inside the "with" block.
# How it works: "with ... as cb:" starts the meter; every model call inside the block is added to cb.usage_metadata.
from langchain_core.callbacks import get_usage_metadata_callback   # the token meter

for question in ["What is the capital of Japan?", "What are the best laptops under 80,000 NPR right now?"]:
    with get_usage_metadata_callback() as cb:
        state = router_workflow.invoke({"question": question})
    print("Question:", question)
    print("Path    :", state["decision"])
    print("Tokens  :", cb.usage_metadata)
    print("Answer  :", state["answer"][:300])
    print()
# expected: the capital question takes "direct" and uses a few hundred tokens;
#           the laptop question takes "search", uses a few thousand tokens, and the answer has a link

**Talk about it:** which path used more tokens? Why? (Hint: count how many times the search path calls the model.)

## Classwork 3 · A support router (about 15 minutes)
**Build it yourself** in the empty cell below, as a LangGraph graph. Use TODO 5 to 7 as your guide, but type every line yourself.

A help desk gets three kinds of messages: **billing** (payments, refunds), **technical** (something does not work), and **other**. Each team gets its own reply step.

```
                      +--"billing"----> billing_reply   --> END
START --> sort_ticket +--"technical"--> technical_reply --> END
                      +--"other"------> other_reply     --> END
```

**Steps**
1. A class `Ticket` with one field, `team`: `"billing"`, `"technical"` or `"other"`, with a description. Make `ticket_router` with structured output.
2. A state `SupportState` with the keys `message`, `team` and `reply`.
3. Four nodes: `sort_ticket` (asks `ticket_router` and saves `team`), and three reply nodes, each one model call with its own instruction that saves `reply`.
4. A gate `pick_team` that turns the team word into the name of a reply node.
5. The graph: START, `sort_ticket`, the gate, one reply node, END. Compile it as `support_router`.

<details><summary><b>Stuck? Open a hint</b></summary>

- The class and the router: like `Route` in TODO 1, with three words.
- A reply node: `def billing_reply(state: SupportState): return {"reply": llm.invoke(f"You are a friendly billing assistant. Reply in 2 sentences: {state['message']}").content}`
- The gate: one `if` per team; the last `return "other_reply"` is the default.
- The graph: like TODO 7, with three choices in the dictionary of `add_conditional_edges`, and an edge from every reply node to `END`.
</details>

In [ ]:
# CLASSWORK 3: build support_router (state keys: message, team, reply).

# Write your code here.

In [ ]:
# CHECK: one message for each team.
for message in ["I was charged twice for my subscription this month.",
                "The app crashes every time I open the camera.",
                "Do you have an office in Pokhara?"]:
    state = support_router.invoke({"message": message})
    print(state["team"].ljust(9), "<-", message)
    print("   ", state["reply"][:200])
# expected teams: billing, technical, other

**If you finish early:** draw `support_router` with `draw_mermaid_png`, then add a fourth team, `sales`, for people who want to buy something: one more word, one more node, one more `if` and one more edge.

---
# Today you learned
- **Structured output** (`with_structured_output`) makes the model fill a class your code can read.
- **A graph** has a **state**, **nodes** (functions) and **edges**. A **conditional edge** (a gate) chooses the next node.
- **Prompt chaining**: steps in a fixed order, with a gate between them. You built it twice:
  - with **LangChain**: `prompt | llm | StrOutputParser()`, short, for steps in a straight line,
  - with **LangGraph**: a state, nodes and a conditional edge, where you can see every step (and, from Day 2, run steps in parallel and loop).
- **Routing**: one step chooses the path, so easy questions take the cheap path.

**Next class: parallel work and the orchestrator-worker pattern.** One question becomes a plan, and many workers research it at the same time.